In [ ]:
# ── MLOps bootstrap (auto-injected by inject_mlops_cell.py) ──────────────────
import os, warnings, mlflow
warnings.filterwarnings("ignore")

SEED = 42
import random, numpy as np
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED)
except ImportError:
    pass
try:
    import tensorflow as tf; tf.random.set_seed(SEED)
except ImportError:
    pass

_nb_name = os.path.basename(os.path.abspath("__file__") if "__file__" in dir() else ".").replace(".ipynb","")
mlflow.set_tracking_uri("sqlite:///" + str(Path(__file__).parent.parent.parent / "mlflow.db")
                        if "__file__" in dir() else "sqlite:///mlflow.db")
_exp = mlflow.set_experiment(_nb_name or "unnamed_notebook")
print(f"MLflow experiment: {_exp.name}")


# 📖 Internal Wiki Assistant

## What We're Building

An assistant that indexes **multiple documentation sources** — Markdown files,
Notion-style exports, and plain text docs — into a single unified chat interface.

## The Problem

Teams scatter knowledge across:
- Engineering wiki (Markdown)
- HR/onboarding docs (HTML/Notion exports)
- Process docs (text files)
- README files in various repos

Users shouldn't need to know *where* something is documented. They should
just ask a question.

## Our Approach

1. **Multi-format loader**: Handle `.md`, `.txt`, `.html`
2. **Source tracking**: Every chunk knows its file, section, and format
3. **Department tagging**: Auto-tag by folder/filename patterns
4. **Unified search**: Search across all docs, or filter by department

## Stack
- **LangChain** — document loaders + retrieval + generation
- **ChromaDB** — vector store with metadata filters
- **Ollama** — local LLM + embeddings

## Step 1 — Imports

In [ ]:
# !pip install langchain langchain-ollama langchain-community chromadb -q

import os
import tempfile
from pathlib import Path
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain.schema import Document

print("All imports successful!")

## Step 2 — Create Sample Wiki Content

We simulate a company wiki with docs across departments.

In [ ]:
# Create a temporary wiki structure
wiki_dir = Path(tempfile.mkdtemp(prefix="wiki_"))

wiki_files = {
    # Engineering docs
    "engineering/deployment-guide.md": """# Deployment Guide

## Production Deployment Process

### Prerequisites
- All tests must pass in CI (GitHub Actions)
- PR must have at least 2 approvals
- No critical or high-severity Snyk vulnerabilities
- Feature flag must be configured in LaunchDarkly

### Deployment Steps
1. Merge PR to `main` branch
2. GitHub Actions automatically builds Docker image
3. Image is pushed to ECR (AWS Elastic Container Registry)
4. ArgoCD detects the new image and rolls out to staging
5. Run smoke tests against staging: `make smoke-test ENV=staging`
6. Promote to production: `argocd app sync prod --prune`
7. Monitor Datadog dashboard for 30 minutes post-deploy

### Rollback Procedure
If issues are detected post-deploy:
1. Run `argocd app rollback prod`
2. This reverts to the previous container image
3. Page the on-call engineer via PagerDuty
4. Create a post-mortem document in Confluence

### Hotfix Process
For critical production issues:
1. Branch from `main` as `hotfix/description`
2. Get 1 approval (reduced from 2 for hotfixes)
3. Merge and deploy — skip staging for P0 incidents
4. Backport fix to any active feature branches
""",
    "engineering/architecture.md": """# System Architecture

## Overview
Our platform is a microservices architecture deployed on AWS EKS.

## Core Services
- **API Gateway** (Kong): Routes and rate-limits all external traffic
- **Auth Service** (Go): Handles authentication, JWT issuance, SSO
- **User Service** (Python/FastAPI): User profiles, preferences, teams
- **Billing Service** (Java/Spring): Subscriptions, invoices, Stripe integration
- **Notification Service** (Node.js): Email, SMS, push, webhooks
- **Analytics Service** (Python): Event processing, dashboards, exports

## Data Stores
- **PostgreSQL** (RDS): Primary database for all transactional data
- **Redis** (ElastiCache): Session cache, rate limiting, feature flags
- **Elasticsearch**: Full-text search, log aggregation
- **S3**: File storage, backups, data exports

## Communication
- Synchronous: gRPC between services, REST for external APIs
- Asynchronous: SQS queues for event processing, SNS for pub/sub

## Environments
- **dev**: Individual developer environments (Tilt + Kind)
- **staging**: Shared environment, auto-deployed from main
- **production**: 3 AZs, auto-scaling, WAF protection
""",
    "engineering/on-call.md": """# On-Call Runbook

## On-Call Rotation
- Schedule: Weekly rotation, Monday 9am to Monday 9am
- Primary + Secondary on-call for each rotation
- Schedule managed in PagerDuty

## Severity Levels

### P0 — Service Down
- Response time: 15 minutes
- All hands on deck
- Status page update within 30 minutes
- Example: API returning 500 for all users

### P1 — Major Feature Broken
- Response time: 30 minutes
- Primary on-call handles
- Example: Payments failing, login broken for SSO users

### P2 — Minor Feature Broken
- Response time: 4 hours (business hours)
- Can wait until morning if after-hours
- Example: Dashboard widget not loading, export CSV formatting

## Common Issues and Quick Fixes

### Database Connection Pool Exhausted
Symptom: 500 errors, logs show "connection pool exhausted"
Fix: `kubectl rollout restart deployment/api-server -n prod`
Root cause: Usually a leaked connection from a long-running query

### Redis Memory Full
Symptom: Slow responses, session timeouts
Fix: `redis-cli FLUSHDB` on the cache cluster (NOT the session cluster)
Prevention: Check TTLs are set on all cache keys

### Certificate Expiry
Symptom: SSL errors in browser
Fix: cert-manager should auto-renew. If not: `kubectl delete certificate && kubectl apply -f cert.yaml`
""",
    # HR docs
    "hr/onboarding.md": """# New Employee Onboarding

## Week 1 Checklist
- [ ] Receive laptop and set up accounts (IT will email credentials)
- [ ] Complete HR paperwork in BambooHR
- [ ] Set up 2FA on all accounts (Google Workspace, GitHub, AWS)
- [ ] Join Slack channels: #general, #team-{your-team}, #random
- [ ] Schedule 1:1s with your manager and skip-level
- [ ] Complete security awareness training (KnowBe4)
- [ ] Read the Engineering Handbook (this wiki)

## Week 2-4: Buddy Program
- You'll be assigned an onboarding buddy from your team
- Buddy will pair-program with you on your first PR
- Weekly check-ins with your buddy for the first month

## Benefits Overview
- Health insurance: Cigna PPO, starts Day 1
- 401(k): Company matches 4%, vests immediately
- PTO: 20 days/year + 10 company holidays
- Learning budget: $2,000/year for conferences, courses, books
- Home office stipend: $1,500 one-time for new hires

## Probation Period
- First 90 days is a probationary period
- Performance review at 30, 60, and 90 days
- Feedback is bidirectional — please share onboarding feedback
""",
    "hr/time-off-policy.md": """# Time Off Policy

## PTO (Paid Time Off)
- All full-time employees: 20 days/year
- Accrual: 1.67 days/month
- Maximum carryover: 5 days to the next year
- Approval: Direct manager via BambooHR
- Notice: 2 weeks for 3+ consecutive days

## Sick Leave
- 10 days/year, separate from PTO
- No advance notice required
- Doctor's note needed for 3+ consecutive sick days
- Can be used for mental health days

## Company Holidays (2025)
1. New Year's Day — Jan 1
2. MLK Day — Jan 20
3. Presidents' Day — Feb 17
4. Memorial Day — May 26
5. Independence Day — Jul 4
6. Labor Day — Sep 1
7. Thanksgiving — Nov 27-28 (2 days)
8. Winter Break — Dec 24-26 (3 days)

## Parental Leave
- Birth/adoption: 16 weeks paid leave
- Non-birth parent: 8 weeks paid leave
- Can be taken in blocks within 12 months

## Bereavement Leave
- Immediate family: 5 days paid
- Extended family: 3 days paid
""",
    # Process docs
    "process/incident-response.txt": """INCIDENT RESPONSE PROCESS

1. DETECTION
   An incident can be detected by:
   - Automated monitoring alerts (Datadog, PagerDuty)
   - Customer reports (via support tickets or Slack)
   - Internal discovery by team members

2. TRIAGE
   The on-call engineer triages the incident:
   - Assign severity level (P0-P3)
   - Create incident channel: #inc-YYYYMMDD-description
   - Page additional engineers if needed
   - Update status page (statuspage.io)

3. INVESTIGATION
   - Check Datadog dashboards for anomalies
   - Review recent deployments in ArgoCD
   - Check error logs in Elasticsearch
   - Identify blast radius (which customers affected?)

4. MITIGATION
   - Implement a quick fix (rollback, config change, restart)
   - Priority is to restore service, not find root cause
   - Communicate status every 30 minutes on status page

5. RESOLUTION
   - Confirm service is restored
   - Notify affected customers
   - Update status page to resolved

6. POST-MORTEM
   Within 48 hours:
   - Write post-mortem document (template in Confluence)
   - Identify root cause and contributing factors
   - Define action items with owners and deadlines
   - Share in #engineering channel
   - No blame — focus on systemic improvements
""",
    "process/code-review.txt": """CODE REVIEW GUIDELINES

PRINCIPLES
- Every change needs a code review before merging
- Review for correctness, maintainability, and security
- Be kind, be specific, be constructive
- Respond to review requests within 4 business hours

PR SIZE GUIDELINES
- Ideal: < 200 lines changed
- Acceptable: 200-500 lines
- Needs justification: 500+ lines (consider splitting)
- Exception: Auto-generated code, migrations, dependency updates

WHAT TO LOOK FOR
1. Correctness: Does it do what it claims?
2. Edge cases: Null values, empty lists, race conditions
3. Tests: Are new behaviors tested? Are edge cases covered?
4. Security: SQL injection, XSS, auth bypass, secrets in code
5. Performance: N+1 queries, unbounded loops, missing indexes
6. Readability: Clear naming, appropriate comments, no magic numbers

APPROVAL REQUIREMENTS
- Regular PRs: 2 approvals from team members
- Hotfixes: 1 approval from any engineer
- Infrastructure changes: 1 approval from platform team
- Security-sensitive: 1 approval from security team

REVIEW ETIQUETTE
- Prefix comments: nit:, suggestion:, question:, blocker:
- Don't block on style if linter passes
- Approve with minor comments when appropriate
- If you can't review in time, reassign promptly
""",
}

# Write files to disk
for filepath, content in wiki_files.items():
    full_path = wiki_dir / filepath
    full_path.parent.mkdir(parents=True, exist_ok=True)
    full_path.write_text(content, encoding="utf-8")

print(f"Created wiki at: {wiki_dir}")
for f in sorted(wiki_files.keys()):
    print(f"  📄 {f}")

## Step 3 — Multi-Format Loader with Auto-Tagging

Load all files, detect format, and auto-tag by department based on
the folder structure.

In [ ]:
def load_wiki(wiki_root: Path) -> list[Document]:
    """Load all supported files from the wiki directory tree."""
    supported = {".md", ".txt", ".text", ".markdown"}
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=600,
        chunk_overlap=80,
        separators=["\n## ", "\n### ", "\n\n", "\n", ". "],
    )
    
    all_docs = []
    for file_path in sorted(wiki_root.rglob("*")):
        if file_path.suffix.lower() not in supported:
            continue
        
        content = file_path.read_text(encoding="utf-8")
        relative = file_path.relative_to(wiki_root)
        
        # Auto-detect department from folder name
        parts = relative.parts
        department = parts[0] if len(parts) > 1 else "general"
        
        # Create document and split
        doc = Document(
            page_content=content,
            metadata={
                "source": str(relative),
                "department": department,
                "format": file_path.suffix.lstrip("."),
                "filename": file_path.name,
            },
        )
        chunks = splitter.split_documents([doc])
        all_docs.extend(chunks)
        print(f"  Loaded {relative} → {len(chunks)} chunks [{department}]")
    
    return all_docs


all_chunks = load_wiki(wiki_dir)
print(f"\nTotal: {len(all_chunks)} chunks")

In [ ]:
embeddings = OllamaEmbeddings(model="nomic-embed-text-v2-moe")
vectorstore = Chroma.from_documents(
    documents=all_chunks,
    embedding=embeddings,
    collection_name="internal_wiki",
)

print(f"Indexed {vectorstore._collection.count()} wiki chunks")

# Show department distribution
from collections import Counter
depts = Counter(c.metadata["department"] for c in all_chunks)
for dept, count in depts.items():
    print(f"  {dept}: {count} chunks")

## Step 4 — Wiki Assistant with Department Filter

In [ ]:
llm = ChatOllama(model="qwen3.5:9b", temperature=0.1)

wiki_prompt = ChatPromptTemplate.from_template(
    """You are an internal wiki assistant for a tech company. Answer questions
using the wiki documentation provided.

GUIDELINES:
- Cite the source file for each piece of information: [source: filename]
- Be specific — include exact commands, URLs, and process steps
- Distinguish between policies (HR), processes (engineering), and guidelines
- If the wiki doesn't cover the question, say so clearly
- For multi-step processes, list steps in order

Wiki Documentation:
{context}

Question: {question}

Answer:"""
)


def ask_wiki(
    question: str,
    department: str | None = None,
) -> None:
    """Ask the wiki assistant. Optionally filter by department."""
    print(f"📖 {question}")
    if department:
        print(f"   (filtered to: {department})")
    print("=" * 60)
    
    # Search with optional filter
    if department:
        results = vectorstore.similarity_search(
            question, k=5, filter={"department": department}
        )
    else:
        results = vectorstore.similarity_search(question, k=5)
    
    # Show sources
    print("\n📋 Sources:")
    seen = set()
    for doc in results:
        src = doc.metadata["source"]
        if src not in seen:
            seen.add(src)
            print(f"  📄 {src} [{doc.metadata['department']}]")
    print()
    
    # Build context
    context = "\n\n---\n\n".join(
        f"[Source: {d.metadata['source']} | Department: {d.metadata['department']}]\n"
        f"{d.page_content}"
        for d in results
    )
    
    chain = wiki_prompt | llm | StrOutputParser()
    answer = chain.invoke({"context": context, "question": question})
    
    print(answer)
    print("\n" + "=" * 60)


print("Wiki assistant ready!")

## Step 5 — Ask the Wiki

In [ ]:
ask_wiki("How do I deploy to production?")

In [ ]:
ask_wiki("What's the PTO policy? How many days do I get?")

In [ ]:
ask_wiki("What should I do if there's a P0 incident?", department="engineering")

In [ ]:
ask_wiki("I'm new to the company. What do I need to do in my first week?")

In [ ]:
# Cross-department query
ask_wiki("What databases and data stores do we use?")

In [ ]:
ask_wiki("What are the code review requirements for hotfixes vs regular PRs?")

## 🧠 Key Concepts Recap

| Concept | What it does |
|---------|-------------|
| **Multi-format loader** | Reads `.md`, `.txt` files from a directory tree |
| **Auto department tagging** | Infers department from folder structure |
| **Department filtering** | ChromaDB metadata filter for scoped search |
| **Source citations** | Every answer references the specific wiki file |
| **Cross-department search** | No filter = search all departments at once |

## 🔧 Customization Ideas

- **Notion API loader**: Use `NotionDBLoader` for live Notion integration
- **Confluence loader**: Use `ConfluenceLoader` for Atlassian wikis
- **Incremental updates**: Watch for file changes and re-index only modified docs
- **Access control**: Filter results based on user's department/role